In [ ]:
#import the kaggle token to access the dataset
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")


In [ ]:
#Import the kaggle Dataset and unzip it
!kaggle datasets download -d mobeenfatimah/fake-news-detection-dataset-6000-news-articles
!unzip fake-news-detection-dataset-6000-news-articles.zip


Dataset URL: https://www.kaggle.com/datasets/mobeenfatimah/fake-news-detection-dataset-6000-news-articles
License(s): CC-BY-SA-4.0
100% 5.77M/5.77M [00:00<00:00, 169MB/s]

Archive:  fake-news-detection-dataset-6000-news-articles.zip
  inflating: news_dataset.csv        


In [ ]:
# import pandas and make it read the file
import pandas as pd

df = pd.read_csv('news_dataset.csv')

In [ ]:
# Check for null values and remove

df.isnull().sum()
df.isnull().values.any()
df.info()
df.dropna()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    6000 non-null   object
 1   text     6000 non-null   object
 2   subject  6000 non-null   object
 3   date     6000 non-null   object
 4   label    6000 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 234.5+ KB


,title,text,subject,date,label
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1
...,...,...,...,...,...
5995,Exiled Venezuelan opposition magistrates resur...,SANTIAGO (Reuters) - A group of opposition-app...,worldnews,"October 19, 2017",1
5996,Plague outbreak in Madagascar kills 20: WHO,NAIROBI (Reuters) - An outbreak of plague has ...,worldnews,"September 29, 2017",1
5997,"Trump to visit Asia in November, North Korea i...",WASHINGTON (Reuters) - Donald Trump will trave...,worldnews,"September 29, 2017",1
5998,Melania Trump calls taped comments by Donald T...,(Reuters) - Melania Trump rose to her husband’...,politicsNews,"October 18, 2016",1


In [ ]:
# clean the code to lowercase "title", "text", and "subject"
df['title'] = df['title'].str.lower()
df['text'] = df['text'].str.lower()
df['subject'] = df['subject'].str.lower()

In [ ]:
# Remove the punctuation in title and text
df['title'] = df['title'].str.replace(r'[^a-zA-Z0-9\s]+', ' ', regex=True)
df['text'] = df['text'].str.replace(r'[^a-zA-Z0-9\s]+', ' ', regex=True)
df['subject'] = df['subject'].str.replace(r'[^a-zA-Z0-9\s]+', '' , regex=True)
print(df)

                                                  title  \
0      watch  trump just told all the anti gay bigot...   
1     china backs u n  call for justice in yemen  u ...   
2     the people s president  trump meets with coal ...   
3     wsj reporter rips into dem candidates for thei...   
4     lawmakers aim to delay u s  ceding control of ...   
...                                                 ...   
5995  exiled venezuelan opposition magistrates resur...   
5996        plague outbreak in madagascar kills 20  who   
5997  trump to visit asia in november  north korea i...   
5998  melania trump calls taped comments by donald t...   
5999   boycottpenzeys  hateful  divisive penzeys spi...   

                                                   text       subject  \
0     a whole lot of evangelical trump voters just d...          news   
1     geneva  reuters    china signaled on wednesday...     worldnews   
2                                                            politics   

In [ ]:
# Drop duplicates and Remove the words stopwords from title and text
df = df.drop_duplicates().copy()
generalwords = ['a', 'an', 'is', 'the', 'to', 'in', 'she', 'it', 'they', 'in', 'on', 'at', 'with']
regexp = r'\b(' + '|'.join(generalwords) + r')\b'
df['title'] = df['title'].str.replace(regexp, ' ', regex = True)
df['text'] = df['text'].str.replace(regexp, ' ', regex = True)


In [ ]:
# Begin the lemmanization process using spacy
import spacy

nlp = spacy.load("en_core_web_sm")

def lemmanization(text):
  doc = nlp(text)
  return " ".join([token.lemma_ for token in doc])

In [ ]:
# Tokenization using .split built in pandas
df['tokens'] = df['text'].str.split()

In [ ]:
df['text_lemmatized'] = df['text'].apply(lemmanization)
display(df.head())

,title,text,subject,date,label,tokens,text_lemmatized
0,watch trump just told all anti gay bigots ...,whole lot of evangelical trump voters just d...,news,"November 13, 2016",0,"[whole, lot, of, evangelical, trump, voters, j...",whole lot of evangelical trump voter just d...
1,china backs u n call for justice yemen u s...,geneva reuters china signaled wednesday ...,worldnews,"September 13, 2017",1,"[geneva, reuters, china, signaled, wednesday, ...",geneva reuters china signal wednesday...
2,people s president trump meets coal worke...,,politics,"Feb 16, 2017",0,[],
3,wsj reporter rips into dem candidates for thei...,wsj s mary kissel nails dem debate ...,politics,"Nov 15, 2015",0,"[wsj, s, mary, kissel, nails, dem, debate, thr...",wsj s mary kissel nail dem debate ...
4,lawmakers aim delay u s ceding control of i...,washington reuters critics of plan for ...,politicsnews,"September 13, 2016",1,"[washington, reuters, critics, of, plan, for, ...",washington reuters critic of plan for...


In [ ]:
!pip install gensim
from gensim.models import Word2Vec
sentences = df['tokens'].tolist()
model = Word2Vec(sentences, vector_size = 100, window = 4, min_count = 1, workers = 1, sg = 0)




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.7 MB/s eta 0:00:00


In [ ]:
import numpy as np

def get_document_embedding(tokens, model):
    valid_words = [word for word in tokens if word in model.wv.key_to_index]

    if not valid_words:
        return np.zeros(model.vector_size)

    word_vectors = np.array([model.wv[word] for word in valid_words])

    return np.mean(word_vectors, axis=0)

df['document_embedding'] = df['tokens'].apply(lambda x: get_document_embedding(x, model))

In [ ]:
from sklearn.model_selection import train_test_split

X = np.array(df['document_embedding'].tolist())
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (4796, 100)
Shape of X_test: (1199, 100)
Shape of y_train: (4796,)
Shape of y_test: (1199,)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

classifier = LogisticRegression(max_iter=1000, random_state=42)

print("Training Logistic Regression model...")
classifier.fit(X_train, y_train)
print("Model training complete.")

y_pred = classifier.predict(X_test)

print("\nModel Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Training Logistic Regression model...
Model training complete.

Model Evaluation:
Accuracy: 0.9583
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       600
           1       0.96      0.95      0.96       599

    accuracy                           0.96      1199
   macro avg       0.96      0.96      0.96      1199
weighted avg       0.96      0.96      0.96      1199



In [ ]:
# Install necessary libraries for web scraping
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_text_from_url(url):
     # Fetches the main text from url
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract text from common content
        paragraphs = soup.find_all('p')
        article_text = ' '.join([p.get_text() for p in paragraphs])

        # Fallback to body text if paragraphs are little
        if not article_text.strip():
            article_text = soup.body.get_text(separator=' ', strip=True)

        return article_text.strip()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL {url}: {e}")
        return ""
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return ""

In [ ]:
import re
import spacy

# Processes the data like in the dataset
def preprocess_new_text(text):
    # Make sure global variables are available
    global generalwords, regexp, nlp, lemmanization

    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9\s]+', ' ', text)

    text = re.sub(regexp, ' ', text)

    lemmatized_text = lemmanization(text)

    tokens = lemmatized_text.split()

    return tokens

In [ ]:
# Predicting whether the article is actually fake news or not
def predict_fake_news(url, word2vec_model, classifier_model):

    print(f"Processing URL: {url}")

    raw_text = fetch_text_from_url(url)
    if not raw_text:
        return "Could not fetch content or content was empty."
    print(f"Fetched content (first 200 chars): {raw_text[:200]}...")

    processed_tokens = preprocess_new_text(raw_text)
    if not processed_tokens:
        return "No valid tokens after preprocessing. Cannot make a prediction."
    print(f"Processed tokens (first 10): {processed_tokens[:10]}...")

    global get_document_embedding
    if 'get_document_embedding' not in globals():
        print("Error: get_document_embedding function not found. Please ensure all previous cells are run.")
        return "Error: Preprocessing function not loaded."

    document_embedding = get_document_embedding(processed_tokens, word2vec_model)
    if np.all(document_embedding == 0):
        print("Warning: Document embedding is all zeros. This might indicate no known words were found.")

    prediction = classifier_model.predict(document_embedding.reshape(1, -1))
    probability = classifier_model.predict_proba(document_embedding.reshape(1, -1))

    result = "Real News" if prediction[0] == 1 else "Fake News"

    print(f"Prediction: {result} (Probability: Fake={probability[0][0]:.4f}, Real={probability[0][1]:.4f})")
    return result

In [ ]:
# Paste in the url to check if fake news or not (the onion is used as an example)
example_url = "https://theonion.com/immunologists-warn-about-rise-of-good-time-resistant-supernarcs/"

if 'model' not in globals() or 'classifier' not in globals():
    print("Error: Word2Vec model or classifier not found. Please ensure all previous cells are run.")
else:
    re.escape(example_url)
    prediction_result = predict_fake_news(url=example_url, word2vec_model=model, classifier_model=classifier)
    print(f"\nFinal Prediction for {example_url}: {prediction_result}")

Processing URL: https://theonion.com/immunologists-warn-about-rise-of-good-time-resistant-supernarcs/
Fetched content (first 200 chars): Become a Member. Get the Paper. Wednesday, August 12, 2026 

81°
 Smoggy with a soupcon of ragweed America’s Finest News Source Wednesday, August 12, 2026 

81°
 Smoggy with a soupcon of ragweed Newsl...
Processed tokens (first 10): ['become', 'member', 'get', 'paper', 'wednesday', 'august', '12', '2026', '81', 'smoggy']...
Prediction: Fake News (Probability: Fake=0.9580, Real=0.0420)

Final Prediction for https://theonion.com/immunologists-warn-about-rise-of-good-time-resistant-supernarcs/: Fake News
